In [27]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
# classification models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

In [28]:
# Read the preprocessed CSV file
df = pd.read_csv('/content/Social_Network_Ads.csv')

# Inspect the DataFrame
print(df.head())

    User ID  Gender  Age  EstimatedSalary  Purchased
0  15624510    Male   19            19000          0
1  15810944    Male   35            20000          0
2  15668575  Female   26            43000          0
3  15603246  Female   27            57000          0
4  15804002    Male   19            76000          0


In [29]:
df = df.drop("User ID", axis=1)

In [30]:
df.head()

,Gender,Age,EstimatedSalary,Purchased
0,Male,19,19000,0
1,Male,35,20000,0
2,Female,26,43000,0
3,Female,27,57000,0
4,Male,19,76000,0


In [31]:
x = df.drop("Purchased", axis=1)
y = df["Purchased"]

In [32]:
# Encode Gender column if present
if "Gender" in df.columns:
    le = LabelEncoder()
    df["Gender"] = le.fit_transform(df["Gender"])

# Feature Selection

#Recursive Feature Elimination

In [33]:
def split_scalar(x,y):
        X_train, X_test, y_train, y_test = train_test_split(x,y, test_size = 0.25, random_state = 0)

        sc = StandardScaler()
        X_train = sc.fit_transform(X_train)
        X_test = sc.transform(X_test)
        return X_train, X_test, y_train, y_test

In [34]:
def evaluate_classification(y_test, y_pred):
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="weighted")
    rec = recall_score(y_test, y_pred, average="weighted")
    f1 = f1_score(y_test, y_pred, average="weighted")

    print("Classification Report:\n", classification_report(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

    return {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1}

In [35]:
# Logistic Regression
def logistic_classifier(X_train, y_train, X_test, y_test):
    model = LogisticRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return evaluate_classification(y_test, y_pred)

In [36]:
# Decision Tree
def decision_tree_classifier(X_train, y_train, X_test, y_test):
    model = DecisionTreeClassifier(random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return evaluate_classification(y_test, y_pred)

In [37]:
# Random Forest
def random_forest_classifier(X_train, y_train, X_test, y_test):
    model = RandomForestClassifier(random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return evaluate_classification(y_test, y_pred)

In [38]:
# Support Vector Machine
def svm_classifier(X_train, y_train, X_test, y_test):
    model = SVC(probability=True, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return evaluate_classification(y_test, y_pred)

In [39]:
# XGBoost
def xgboost_classifier(X_train, y_train, X_test, y_test):
    model = XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return evaluate_classification(y_test, y_pred)

In [42]:
from sklearn.feature_selection import RFE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

def rfeFeature_classification(df_encoded, n):
    """
    Performs Recursive Feature Elimination (RFE) on a DataFrame with encoded features
    and evaluates classification models using the selected features.

    Args:
        df_encoded (pd.DataFrame): The input DataFrame containing encoded features and the target.
        n (int): The number of features to select.

    Returns:
        tuple: A tuple containing:
            - rfelist (list): A list of DataFrames with selected features.
            - colnames_list (list): A list of lists containing the names of selected columns for each model.
            - scores (list): A list of dictionaries containing accuracy and F1 scores for each model.
    """
    x = df_encoded.drop("Purchased", axis=1)
    y = df_encoded["Purchased"]

    # Explicitly convert columns to numeric, coercing errors
    x = x.apply(pd.to_numeric, errors='coerce')

    # Drop any columns that could not be converted to numeric (if any)
    x = x.dropna(axis=1)


    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

    # Scale the data
    sc = StandardScaler()
    X_train_scaled = sc.fit_transform(X_train)
    X_test_scaled = sc.transform(X_test)

    rfelist = []
    colnames_list = []
    scores = []

    # Models for classification
    log_reg = LogisticRegression(max_iter=1000, random_state=0)
    dec = DecisionTreeClassifier(random_state=0)
    rf = RandomForestClassifier(n_estimators=50, random_state=0)
    xgb = XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=0)

    rfemodellist = [log_reg, dec, rf, xgb]

    for model in rfemodellist:
        # Apply RFE on the training data
        rfe = RFE(estimator=model, n_features_to_select=n)
        rfe_fit = rfe.fit(X_train_scaled, y_train)

        # Transform both training and test data using the selected features
        X_train_rfe = rfe_fit.transform(X_train_scaled)
        X_test_rfe = rfe_fit.transform(X_test_scaled)

        rfelist.append(pd.DataFrame(X_train_rfe)) # Append training data transformed by RFE

        # Get the column names selected by RFE (based on original column names)
        selected_columns = [col for col, selected in zip(x.columns, rfe.support_) if selected]
        colnames_list.append(selected_columns)

        # Fit the model on the RFE selected training data
        model.fit(X_train_rfe, y_train)
        y_pred = model.predict(X_test_rfe)

        # Accuracy & F1
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average="weighted")

        scores.append({"Accuracy": acc, "F1": f1})

    return rfelist, colnames_list, scores

# Example usage:
rfelist, colnames_list, scores = rfeFeature_classification(df, 3)



/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [07:17:50] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [07:17:50] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [43]:
# Print results
for model_name, selected_columns, score in zip(
    ["Logistic", "Decision", "RandomForest", "XGBoost"], colnames_list, scores
):
    print(f"Model: {model_name}")
    print("Selected Columns:", selected_columns)
    print(f"Accuracy: {score['Accuracy']:.4f}, F1: {score['F1']:.4f}\n")

Model: Logistic
Selected Columns: ['Gender', 'Age', 'EstimatedSalary']
Accuracy: 0.8875, F1: 0.8846

Model: Decision
Selected Columns: ['Gender', 'Age', 'EstimatedSalary']
Accuracy: 0.8375, F1: 0.8352

Model: RandomForest
Selected Columns: ['Gender', 'Age', 'EstimatedSalary']
Accuracy: 0.9125, F1: 0.9121

Model: XGBoost
Selected Columns: ['Gender', 'Age', 'EstimatedSalary']
Accuracy: 0.8750, F1: 0.8750

